In [1]:
"""
This code loads earnings announcement data and historical stock price data.

It defines the observation length, future prediction horizon, upward and
downward return thresholds, column positions, and desired price channels such
as open, high, low, close, adjusted close, and volume. The earnings and price
dates are converted to datetime format, and the earnings rows are shuffled for
processing.

The build_multivariate_dataset_using_adjclose function creates samples around
each valid earnings event. For each ticker and earnings date, it determines the
effective trading date based on whether the earnings release happened before
or after the market opened. It then extracts:
- an observed price window before the earnings event,
- a future price window after the earnings event,
- metadata containing the ticker and event date.

The function resolves price column names flexibly, skips invalid release times,
handles missing tickers or insufficient data coverage, removes samples with NaN
values when requested, and uses tqdm to show progress while processing earnings
dates.

After all valid samples are collected, the code calculates the net future return
using the adjusted close or close price. Each sample is assigned a label:
- 1 for an upward move above the positive threshold,
- -1 for a downward move below the negative threshold,
- 0 for a relatively flat move between the thresholds.

The code then filters out the flat class, keeping only samples with a meaningful
upward or downward movement. If no samples remain after filtering, it raises an
error suggesting that the return thresholds or data coverage should be checked.

Finally, the code prints the number of remaining samples, the class counts for
downward and upward movements, the shapes of the observed and future arrays,
and the resolved channel order so the user can verify that the binary stock
movement dataset has been prepared correctly.
"""
import numpy as np
import pandas as pd
import pathlib
import os
import random
import torch
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from MyModels import ModernTCNClassifier,   ModernTCNForecastThenClassifyOfficial, BayesianModernTCNForecastThenClassifyJoint, RocketMultivariateLengths
from tensorflow.keras.utils import to_categorical 
from sklearn.linear_model import RidgeClassifierCV
from tqdm import tqdm
from sklearn.preprocessing import label_binarize
from aeon.classification.convolution_based import MultiRocketHydraClassifier
#https://www.kaggle.com/datasets/tsaustin/us-historical-stock-prices-with-earnings-data
FILE_EARNINGS = '/Users/user/Downloads/earnings_latest.csv'
FILE_PRICES   = '/Users/user/Downloads/stock_prices_latest.csv'
OBS_LEN       = 40
FUTURE_LEN    = 5
UP_THRESH     = 0.01    
DOWN_THRESH   = -0.01   
PRICE_TICKER_COL = 0
PRICE_DATE_COL   = 1
EARN_TICKER_COL = 0
EARN_DATE_COL   = 1
EARN_TIME_COL   = 5  
DESIRED_CHANNELS = ("open", "high", "low", "close", "close_adjusted", "volume")
df = pd.read_csv(FILE_EARNINGS)
prices = pd.read_csv(FILE_PRICES)
df.iloc[:, EARN_DATE_COL] = pd.to_datetime(df.iloc[:, EARN_DATE_COL])
prices.iloc[:, PRICE_DATE_COL] = pd.to_datetime(prices.iloc[:, PRICE_DATE_COL])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
def build_multivariate_dataset_using_adjclose(
    df,
    prices,
    OBS_LEN,
    FUTURE_LEN,
    UP_THRESH,
    DOWN_THRESH,
    PRICE_TICKER_COL,
    PRICE_DATE_COL,
    EARN_TICKER_COL,
    EARN_DATE_COL,
    EARN_TIME_COL,
    desired_channels=DESIRED_CHANNELS,
    skip_on_nan=True,
):
    def _norm(s):
        return str(s).lower().replace(" ", "").replace("_", "").replace("-", "")
    def _match_col(cols, target):
        t = _norm(target)
        for c in cols:
            if _norm(c) == t:
                return c
        for c in cols:
            k = _norm(c)
            if t in k or k in t:
                return c
        return None
    price_cols = list(prices.columns)
    ticker_col_name = prices.columns[PRICE_TICKER_COL]
    date_col_name = prices.columns[PRICE_DATE_COL]
    resolved_channels = []
    for ch in desired_channels:
        found = _match_col(price_cols, ch)
        if found is None:
            raise RuntimeError(f"Could not find expected price column matching '{ch}' in: {price_cols}")
        resolved_channels.append(found)
    close_pref_candidates = ["close_adjusted", "adjclose", "adj_close", "adjustedclose", "close"]
    close_match = None
    for cand in close_pref_candidates:
        c = _match_col(price_cols, cand)
        if c is not None and c in resolved_channels:
            close_match = c
            break
    if close_match is None:
        raise RuntimeError("Could not locate an adjusted/close column among resolved channels for label calculation.")
    ticker_prices_dict = {ticker: group.reset_index(drop=True) for ticker, group in prices.groupby(prices.columns[PRICE_TICKER_COL])}
    observed_windows = []
    future_windows = []
    meta_rows = []
    with tqdm(total=len(df), desc="Processing earnings dates") as pbar:
        for _, row in df.iterrows():
            ticker = row.iloc[EARN_TICKER_COL]
            earnings_date = row.iloc[EARN_DATE_COL]
            release_time = row.iloc[EARN_TIME_COL]
            if pd.isna(release_time) or release_time not in ("pre", "post"):
                pbar.update(1); continue
            t0_calendar = earnings_date if release_time == "pre" else (earnings_date + pd.Timedelta(days=1))
            tp = ticker_prices_dict.get(ticker)
            if tp is None or tp.empty:
                pbar.update(1); continue
            try:
                tdates = tp[date_col_name].values.astype("datetime64[ns]")
            except Exception:
                tp = tp.copy()
                tp[date_col_name] = pd.to_datetime(tp[date_col_name])
                tdates = tp[date_col_name].values.astype("datetime64[ns]")
            idxs = np.where(tdates >= np.datetime64(t0_calendar))[0]
            if idxs.size == 0:
                pbar.update(1); continue
            t0_idx = int(idxs[0])
            if t0_idx - OBS_LEN < 0 or t0_idx + FUTURE_LEN >= len(tp):
                pbar.update(1); continue
            try:
                feat_df = tp.loc[:, resolved_channels].astype(float).reset_index(drop=True)
            except Exception:
                pbar.update(1); continue
            feats = feat_df.values  
            observed = feats[t0_idx - OBS_LEN : t0_idx]                      
            future_seq = feats[t0_idx : t0_idx + FUTURE_LEN + 1]              
            if observed.shape[0] != OBS_LEN or future_seq.shape[0] != (FUTURE_LEN + 1):
                pbar.update(1); continue
            if skip_on_nan and (np.isnan(observed).any() or np.isnan(future_seq).any()):
                pbar.update(1); continue
            observed_windows.append(observed)
            future_windows.append(future_seq)
            meta_rows.append((ticker, pd.Timestamp(tdates[t0_idx]).normalize()))
            pbar.update(1)
    if len(observed_windows) == 0:
        raise RuntimeError("No valid (observed, future) pairs were found. Check column indices and data coverage.")
    X_observed = np.array(observed_windows)            
    future_windows = np.array(future_windows)          
    meta_rows = np.array(meta_rows, dtype=object)
    close_idx = resolved_channels.index(close_match)
    t0_close = future_windows[:, 0, close_idx]
    tF_close = future_windows[:, FUTURE_LEN, close_idx]
    with np.errstate(divide='ignore', invalid='ignore'):
        net_ret = np.where(t0_close != 0, (tF_close - t0_close) / t0_close, 0.0)
    labels = np.zeros_like(net_ret, dtype=int)
    labels[net_ret >= UP_THRESH] = 1
    labels[net_ret <= DOWN_THRESH] = -1
    Y_future_seq = future_windows[:, 1:, :] 
    return X_observed, Y_future_seq, labels, meta_rows, resolved_channels
X_observed, Y_future_seq, labels, meta_rows, resolved_channels = build_multivariate_dataset_using_adjclose(
    df=df,
    prices=prices,
    OBS_LEN=OBS_LEN,
    FUTURE_LEN=FUTURE_LEN,
    UP_THRESH=UP_THRESH,
    DOWN_THRESH=DOWN_THRESH,
    PRICE_TICKER_COL=PRICE_TICKER_COL,
    PRICE_DATE_COL=PRICE_DATE_COL,
    EARN_TICKER_COL=EARN_TICKER_COL,
    EARN_DATE_COL=EARN_DATE_COL,
    EARN_TIME_COL=EARN_TIME_COL,
    desired_channels=DESIRED_CHANNELS,
    skip_on_nan=True,
)
mask = (labels == -1) | (labels == 1)
if not np.any(mask):
    raise RuntimeError(
        "After filtering, no samples remain. "
        "Consider relaxing UP_THRESH/DOWN_THRESH or checking your data coverage."
    )
X_observed = X_observed[mask]
Y_future_seq = Y_future_seq[mask]
labels = labels[mask]
meta_rows = meta_rows[mask]
n = labels.shape[0]
print(f"Total samples (|return| >= {max(UP_THRESH, -DOWN_THRESH):.2%}): {n}")
print(f"Class counts ->  -1 (down): {int((labels == -1).sum())}   1 (up): {int((labels == 1).sum())}")
print("X_observed shape:", X_observed.shape)      
print("Y_future_seq shape:", Y_future_seq.shape)  
print("labels shape:", labels.shape)
print("channel order (resolved):", resolved_channels)

/opt/anaconda3/lib/python3.12/site-packages/aeon/base/__init__.py:24: FutureWarning: The aeon package will soon be releasing v1.0.0 with the removal of legacy modules and interfaces such as BaseTransformer and BaseForecaster. This will contain breaking changes. See aeon-toolkit.org for more information. Set aeon.AEON_DEPRECATION_WARNING or the AEON_DEPRECATION_WARNING environmental variable to 'False' to disable this warning.
  warnings.warn(
Processing earnings dates: 100%|███████| 168603/168603 [09:16<00:00, 302.82it/s]


Total samples (|return| >= 1.00%): 26608
Class counts ->  -1 (down): 14484   1 (up): 12124
X_observed shape: (26608, 40, 6)
Y_future_seq shape: (26608, 5, 6)
labels shape: (26608,)
channel order (resolved): ['open', 'high', 'low', 'close', 'close_adjusted', 'volume']


In [2]:
"""
This code chronologically splits the filtered stock-event time-series dataset
into training and testing sets for a binary up/down classification task.

It first tries to extract the earnings event dates from meta_rows and uses them
to sort all samples in time order. If the dates cannot be parsed, it falls back
to the existing sample order. The observed windows, future sequences, and labels
are then reordered consistently using this ordering.

The original movement labels are converted from:
- -1 for downward movement,
- 1 for upward movement,

into binary class labels:
- 0 for downward movement,
- 1 for upward movement.

The dataset is then split using TEST_SIZE, with the earliest samples used for
training and the latest samples used for testing. This preserves chronological
ordering and avoids training on future samples before evaluating on earlier
ones.

The code prints the shapes of the resulting training and testing arrays to
verify the split. It also selects the best available PyTorch compute device,
prioritising Apple Silicon MPS, then CUDA, and finally CPU.

Finally, the to_int helper function ensures that labels are represented as
integer class labels. If the labels are one-hot encoded, it converts them using
argmax; otherwise, it casts the existing labels directly to integers.
"""
TEST_SIZE = 0.2
try:
    t0_dates = pd.to_datetime([d for _, d in meta_rows])
    order = np.argsort(t0_dates.values)
except Exception:
    order = np.arange(labels.shape[0])
X_obs_all  = X_observed[order]    
X_fore_all = Y_future_seq[order]  
y_all      = labels[order] 
y_all = ((y_all + 1) // 2).astype(int)
N = y_all.shape[0]
split_idx = int(np.floor(N * (1 - TEST_SIZE)))
X_observed_train = X_obs_all[:split_idx]
X_forecast_train  = X_fore_all[:split_idx]
y_train       = y_all[:split_idx]
X_observed_test  = X_obs_all[split_idx:]
X_forecast_test  = X_fore_all[split_idx:]
y_test       = y_all[split_idx:]
print("Train shapes:", X_observed_train.shape, X_forecast_train.shape, y_train.shape)
print("Test shapes: ", X_observed_test.shape,  X_forecast_test.shape,  y_test.shape)
device = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using device: {device}")
def to_int(y):
    y = np.asarray(y)
    return np.argmax(y, axis=1) if (y.ndim == 2 and y.shape[1] > 1) else y.astype(int)  

Train shapes: (21286, 40, 6) (21286, 5, 6) (21286,)
Test shapes:  (5322, 40, 6) (5322, 5, 6) (5322,)
Using device: mps


In [5]:
"""
This code trains and evaluates a ModernTCNClassifier multiple times using
different random seeds to measure how stable the model performance is across
repeated runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

For each seed, the code:
- sets the random seed,
- creates a new ModernTCNClassifier with fixed training hyperparameters,
- trains the model on the observed training sequences,
- predicts class labels and class probabilities on the observed test sequences.
Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
results = []  
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training ModernTCNClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNClassifier(
        max_epochs=50,       
        batch_size=512,      
        lr=1e-3,
        weight_decay=1e-4,
        device=device,       
        verbose=True,
    )
    clf.fit(X_observed_train, y_train)
    y_pred = clf.predict(X_observed_test)          
    proba  = clf.predict_proba(X_observed_test)    
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":auprc0 })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNClassifier - seed 0
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 42/42 [00:01<00:00, 34.91it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.5041    0.9464    0.6578      2685
           1     0.4875    0.0520    0.0939      2637

    accuracy                         0.5032      5322
   macro avg     0.4958    0.4992    0.3758      5322
weighted avg     0.4959    0.5032    0.3784      5322

AUROC (seed 0): 0.4864
AUPRC (seed 0): 0.4877
AUPRC class 0 (seed 0): 0.4897

Training ModernTCNClassifier - seed 1
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 42/42 [00:01<00:00, 35.15it/s]



Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.5091    0.9322    0.6585      2685
           1     0.5506    0.0846    0.1466      2637

    accuracy                         0.5122      5322
   macro avg     0.5298    0.5084    0.4026      5322
weighted avg     0.5296    0.5122    0.4049      5322

AUROC (seed 1): 0.5057
AUPRC (seed 1): 0.5031
AUPRC class 0 (seed 1): 0.5060

Training ModernTCNClassifier - seed 2
[ModernTCN] Using device: mps


...


Classification report (seed 2):
              precision    recall  f1-score   support

           0     0.5069    0.9076    0.6505      2685
           1     0.5175    0.1009    0.1688      2637

    accuracy                         0.5079      5322
   macro avg     0.5122    0.5043    0.4097      5322
weighted avg     0.5121    0.5079    0.4118      5322

AUROC (seed 2): 0.4815
AUPRC (seed 2): 0.4879
AUPRC class 0 (seed 2): 0.4901

Training ModernTCNClassifier - seed 3
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 42/42 [00:01<00:00, 36.11it/s]



Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.5040    0.9039    0.6472      2685
           1     0.4911    0.0944    0.1584      2637

    accuracy                         0.5028      5322
   macro avg     0.4976    0.4992    0.4028      5322
weighted avg     0.4976    0.5028    0.4050      5322

AUROC (seed 3): 0.4918
AUPRC (seed 3): 0.4963
AUPRC class 0 (seed 3): 0.4939

Training ModernTCNClassifier - seed 4
[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 42/42 [00:01<00:00, 36.14it/s]



Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.5063    0.8842    0.6439      2685
           1     0.5087    0.1221    0.1969      2637

    accuracy                         0.5066      5322
   macro avg     0.5075    0.5031    0.4204      5322
weighted avg     0.5075    0.5066    0.4224      5322

AUROC (seed 4): 0.4930
AUPRC (seed 4): 0.4937
AUPRC class 0 (seed 4): 0.4937

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.4864  AUPRC=0.4877AUPRC0=0.4897
Seed 1: AUROC=0.5057  AUPRC=0.5031AUPRC0=0.5060
Seed 2: AUROC=0.4815  AUPRC=0.4879AUPRC0=0.4901
Seed 3: AUROC=0.4918  AUPRC=0.4963AUPRC0=0.4939
Seed 4: AUROC=0.4930  AUPRC=0.4937AUPRC0=0.4937


In [7]:
"""
This code trains and evaluates the ModernTCNForecastThenClassify model
across five different random seeds to assess how consistent its performance is
under different initialisations.

For each seed, the code resets the random state for reproducibility, creates a
new forecast-then-classify model, and provides it with the future training
sequences needed for the forecasting stage. The model is then trained using the
observed training sequences and their binary class labels.

After training, the model predicts class probabilities for the observed test
sequences. These probabilities are converted into predicted class labels using
argmax.

Performance metrics are then computed and printed. 

"""
results = []  
for seed in range(5): 
    print("\n" + "="*80)
    print(f"Training ModernTCNForecastThenClassifyOfficial - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = ModernTCNForecastThenClassifyOfficial(
        X_forecast_train=X_forecast_train,
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    proba = clf.predict_proba(X_observed_test.astype(np.float32))   
    y_pred = np.argmax(proba, axis=1)
    y_true = y_test.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()  
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan,"auprc0":np.nan })
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training ModernTCNForecastThenClassifyOfficial - seed 0
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 42/42 [00:01<00:00, 23.69it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 42/42 [00:01<00:00, 35.46it/s]



Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.5055    0.9006    0.6476      2685
           1     0.5046    0.1031    0.1713      2637

    accuracy                         0.5054      5322
   macro avg     0.5051    0.5019    0.4094      5322
weighted avg     0.5051    0.5054    0.4116      5322

AUROC (seed 0): 0.4898
AUPRC (seed 0): 0.4940
AUPRC class 0 (seed 0): 0.4976

Training ModernTCNForecastThenClassifyOfficial - seed 1
[TwoStage-ModernTCN] Using device: mps


...


Classification report (seed 3):
              precision    recall  f1-score   support

           0     0.5044    0.9169    0.6508      2685
           1     0.4943    0.0827    0.1417      2637

    accuracy                         0.5036      5322
   macro avg     0.4994    0.4998    0.3962      5322
weighted avg     0.4994    0.5036    0.3985      5322

AUROC (seed 3): 0.4909
AUPRC (seed 3): 0.4937
AUPRC class 0 (seed 3): 0.4966

Training ModernTCNForecastThenClassifyOfficial - seed 4
[TwoStage-ModernTCN] Using device: mps


[ModernTCN-forecast] Epoch 20/20: 100%|█████████| 42/42 [00:01<00:00, 23.96it/s]


[ModernTCN] Using device: mps


[ModernTCN] Epoch 50/50: 100%|██████████████████| 42/42 [00:01<00:00, 35.83it/s]



Classification report (seed 4):
              precision    recall  f1-score   support

           0     0.5063    0.9143    0.6517      2685
           1     0.5137    0.0922    0.1563      2637

    accuracy                         0.5070      5322
   macro avg     0.5100    0.5032    0.4040      5322
weighted avg     0.5100    0.5070    0.4062      5322

AUROC (seed 4): 0.4913
AUPRC (seed 4): 0.4957
AUPRC class 0 (seed 4): 0.4923

################################################################################
Summary over seeds
################################################################################
Seed 0: AUROC=0.4898  AUPRC=0.4940AUPRC0=0.4976
Seed 1: AUROC=0.4994  AUPRC=0.4985AUPRC0=0.4992
Seed 2: AUROC=0.4955  AUPRC=0.4983AUPRC0=0.4979
Seed 3: AUROC=0.4909  AUPRC=0.4937AUPRC0=0.4966
Seed 4: AUROC=0.4913  AUPRC=0.4957AUPRC0=0.4923


In [8]:
"""
This code trains and evaluates the ForeClassNet model across five different 
random seeds, using Monte Carlo dropout to estimate prediction probabilities.

The MC_SAMPLES value controls how many stochastic forward passes are used during
Bayesian prediction. For each seed, the code resets the random state for
reproducibility, creates a new ForeClassNet model, and trains
it using the observed training sequences, future forecast training sequences,
and binary class labels.

After training, predict_proba_bayesian is called on the observed test sequences.
This returns the mean predicted class probabilities across the Monte Carlo
samples.

The mean probabilities are converted into predicted class labels using argmax.
Performance metrics are then computed and printed. 
"""
MC_SAMPLES = 100  
results = []
for seed in range(5):  
    print("\n" + "="*80)
    print(f"Training BayesianModernTCNForecastThenClassifyJoint - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = BayesianModernTCNForecastThenClassifyJoint(
        X_forecast_train=X_forecast_train.astype(np.float32),
        max_epochs_forecast=20,
        max_epochs_class=50,
        batch_size=512,          
        lr_forecast=1e-3,
        lr_class=1e-3,
        weight_decay=1e-4,
        device=device,
        verbose=True,
        l=1e-2,
        tau=1.0,
    )
    clf.fit(X_observed_train.astype(np.float32), y_train.astype(int))
    mean_proba, std_proba = clf.predict_proba_bayesian(
        X_observed_test.astype(np.float32),
        n_samples=MC_SAMPLES,
    )  
    y_true = y_test.astype(int)
    y_pred = np.argmax(mean_proba, axis=1)
    print("\nClassification Report (MC dropout, {} samples, seed {}):".format(MC_SAMPLES, seed))
    print(classification_report(y_true, y_pred, digits=4))
    print("-"*72)
    if mean_proba.shape[1] >= 2:
        y_score = mean_proba[:, 1].ravel()
        auroc = roc_auc_score(y_true, y_score)
        auprc = average_precision_score(y_true, y_score)
        y_score_min = mean_proba[:, 0].ravel() 
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auroc:.4f}")
        print(f"AUPRC (MC mean over {MC_SAMPLES} samples, seed {seed}) : {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append({"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0":auprc0})
    else:
        print(f"Warning: only one probability column (shape={mean_proba.shape}); skipping AUROC/AUPRC.")
        results.append({"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0":np.nan})
print("\n" + "#"*80)
print(f"Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC={MC_SAMPLES})")
print("#"*80)
for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}"
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training BayesianModernTCNForecastThenClassifyJoint - seed 0
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 42/42 [00:10<00:00,  4.17it/s]



Classification Report (MC dropout, 100 samples, seed 0):
              precision    recall  f1-score   support

           0     0.5048    0.9750    0.6652      2685
           1     0.5074    0.0262    0.0498      2637

    accuracy                         0.5049      5322
   macro avg     0.5061    0.5006    0.3575      5322
weighted avg     0.5061    0.5049    0.3603      5322

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 0) : 0.5006
AUPRC (MC mean over 100 samples, seed 0) : 0.4989
AUPRC class 0 (seed 0): 0.5035

Training BayesianModernTCNForecastThenClassifyJoint - seed 1
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 42/42 [00:10<00:00,  4.15it/s]



Classification Report (MC dropout, 100 samples, seed 1):
              precision    recall  f1-score   support

           0     0.5042    0.9542    0.6598      2685
           1     0.4896    0.0447    0.0820      2637

    accuracy                         0.5036      5322
   macro avg     0.4969    0.4995    0.3709      5322
weighted avg     0.4970    0.5036    0.3735      5322

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 1) : 0.4944
AUPRC (MC mean over 100 samples, seed 1) : 0.4950
AUPRC class 0 (seed 1): 0.4982

Training BayesianModernTCNForecastThenClassifyJoint - seed 2
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


...


Classification Report (MC dropout, 100 samples, seed 2):
              precision    recall  f1-score   support

           0     0.5050    0.9534    0.6603      2685
           1     0.5059    0.0485    0.0886      2637

    accuracy                         0.5051      5322
   macro avg     0.5055    0.5010    0.3744      5322
weighted avg     0.5055    0.5051    0.3770      5322

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 2) : 0.5057
AUPRC (MC mean over 100 samples, seed 2) : 0.5045
AUPRC class 0 (seed 2): 0.5052

Training BayesianModernTCNForecastThenClassifyJoint - seed 3
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 42/42 [00:10<00:00,  4.15it/s]



Classification Report (MC dropout, 100 samples, seed 3):
              precision    recall  f1-score   support

           0     0.5051    0.9702    0.6644      2685
           1     0.5152    0.0322    0.0607      2637

    accuracy                         0.5054      5322
   macro avg     0.5101    0.5012    0.3625      5322
weighted avg     0.5101    0.5054    0.3652      5322

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 3) : 0.5052
AUPRC (MC mean over 100 samples, seed 3) : 0.5054
AUPRC class 0 (seed 3): 0.5094

Training BayesianModernTCNForecastThenClassifyJoint - seed 4
[BayesianTwoStage-ModernTCN-Joint] Using device: mps


[BayesianModernTCN-Joint] Epoch 50/50: 100%|████| 42/42 [00:10<00:00,  4.13it/s]



Classification Report (MC dropout, 100 samples, seed 4):
              precision    recall  f1-score   support

           0     0.5044    0.9590    0.6611      2685
           1     0.4931    0.0406    0.0750      2637

    accuracy                         0.5039      5322
   macro avg     0.4987    0.4998    0.3680      5322
weighted avg     0.4988    0.5039    0.3707      5322

------------------------------------------------------------------------
AUROC (MC mean over 100 samples, seed 4) : 0.5087
AUPRC (MC mean over 100 samples, seed 4) : 0.5048
AUPRC class 0 (seed 4): 0.5097

################################################################################
Summary over seeds (BayesianModernTCNForecastThenClassifyJoint, MC=100)
################################################################################
Seed 0: AUROC=0.5006  AUPRC=0.4989AUPRC0=0.5035
Seed 1: AUROC=0.4944  AUPRC=0.4950AUPRC0=0.4982
Seed 2: AUROC=0.5057  AUPRC=0.5045AUPRC0=0.5052
Seed 3: AUROC=0.5052  AUPRC=0.50

In [3]:
"""
This code trains and evaluates a MultiRocketHydraClassifier across five
different random seeds to assess how stable its performance is across repeated
runs.

The set_seed function configures Python, NumPy, and PyTorch random seeds so
that each run is as reproducible as possible. It also enables deterministic
PyTorch behaviour and disables cuDNN benchmarking to reduce randomness during
GPU-based training.

The _ensure_labels helper function converts the target labels into a
one-dimensional integer array. If the labels are one-hot encoded, it converts
them into class indices using argmax; otherwise, it flattens the labels and
casts them to integers.

The observed training and testing arrays are transposed into the shape expected
by the aeon/sktime-style classifier: samples, channels, and time steps. The
labels are then prepared for both the training and test sets.

For each seed, the code creates and trains a new MultiRocketHydraClassifier on
the transformed training data. It then predicts class labels and class
probabilities for the transformed test data. Performance metrics are then computed and printed. 
"""
SEED_BASE = 0  
def set_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8" 
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
def _ensure_labels(y):
    y = np.asarray(y)
    if y.ndim > 1 and y.shape[-1] > 1:
        # one-hot -> class indices
        return y.argmax(axis=1).astype(int)
    return y.ravel().astype(int)
X_train_sktime = X_observed_train.transpose(0, 2, 1)
X_test_sktime  = X_observed_test.transpose(0, 2, 1)
y_train_lab = _ensure_labels(y_train)
y_test_lab  = _ensure_labels(y_test)
results = []  
for seed in range(SEED_BASE, SEED_BASE + 5):
    print("\n" + "="*80)
    print(f"Training MultiRocketHydraClassifier - seed {seed}")
    print("="*80)
    set_seed(seed)
    clf = MultiRocketHydraClassifier(random_state=seed)
    clf.fit(X_train_sktime, y_train_lab)
    y_pred = clf.predict(X_test_sktime)
    proba  = clf.predict_proba(X_test_sktime)
    y_true = y_test_lab.astype(int)
    print("\nClassification report (seed {}):".format(seed))
    print(classification_report(y_true, y_pred, digits=4))
    if proba.shape[1] >= 2:
        y_score = proba[:, 1].ravel()
        auroc  = roc_auc_score(y_true, y_score)
        auprc  = average_precision_score(y_true, y_score)
        y_score_min = proba[:, 0].ravel()
        auprc0 = average_precision_score(y_true, y_score_min, pos_label=0)
        print(f"AUROC (seed {seed}): {auroc:.4f}")
        print(f"AUPRC (seed {seed}): {auprc:.4f}")
        print(f"AUPRC class 0 (seed {seed}): {auprc0:.4f}")
        results.append(
            {"seed": seed, "auroc": auroc, "auprc": auprc, "auprc0": auprc0}
        )
    else:
        print("Warning: only one probability column, skipping AUROC/AUPRC.")
        results.append(
            {"seed": seed, "auroc": np.nan, "auprc": np.nan, "auprc0": np.nan}
        )
print("\n" + "#"*80)
print("Summary over seeds")
print("#"*80)

for r in results:
    print(
        f"Seed {r['seed']}: "
        f"AUROC={r['auroc']:.4f}  "
        f"AUPRC={r['auprc']:.4f}  "
        f"AUPRC0={r['auprc0']:.4f}"
    )


Training MultiRocketHydraClassifier - seed 0

Classification report (seed 0):
              precision    recall  f1-score   support

           0     0.4833    0.4525    0.4674      2685
           1     0.4765    0.5074    0.4915      2637

    accuracy                         0.4797      5322
   macro avg     0.4799    0.4800    0.4794      5322
weighted avg     0.4799    0.4797    0.4793      5322

AUROC (seed 0): 0.4800
AUPRC (seed 0): 0.4859
AUPRC class 0 (seed 0): 0.4949

Training MultiRocketHydraClassifier - seed 1

Classification report (seed 1):
              precision    recall  f1-score   support

           0     0.4974    0.4957    0.4965      2685
           1     0.4883    0.4900    0.4891      2637

    accuracy                         0.4929      5322
   macro avg     0.4928    0.4928    0.4928      5322
weighted avg     0.4929    0.4929    0.4929      5322

AUROC (seed 1): 0.4928
AUPRC (seed 1): 0.4920
AUPRC class 0 (seed 1): 0.5010

Training MultiRocketHydraClassifi